# Multiverse Hybrid v3.0 — Stage 1 All-Market Ticket Probability 2000 v2

iPhone/Colab向けの実行安定化版です。

- 既存Stage 1 PASS成果物が正しいSHAで揃っていれば再計算しません
- `files.download()` を使用しません
- 成果物はGoogle Driveへ保存します
- RESULT / PAYOUT / Settlement / EV / ROI は読みません


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, shutil, hashlib, json, zipfile

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE1_PL_PROBABILITY_v1'
STRUCT=OUT/'DEV2000_MARKET_STRUCTURE_ONLY_v1.jsonl'
PROBS=OUT/'DEV2000_ALL_MARKET_TICKET_PROBABILITIES_PL_v1.jsonl'
QUALITY=OUT/'STAGE1_PL_TICKET_PROBABILITY_QUALITY_v1.json'
RECEIPT=OUT/'STAGE1_PL_RECEIPT_v1.json'
LOG=OUT/'STAGE1_PL_RUN_LOG_v1.txt'
ZIP=OUT/'MULTIVERSE_ALL_MARKET_STAGE1_PL_PROBABILITY_v1_ARTIFACT.zip'

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''): h.update(c)
    return h.hexdigest()

# Fast path: do NOT recompute if the already-produced Stage1 output is valid.
if RECEIPT.is_file() and QUALITY.is_file() and PROBS.is_file() and STRUCT.is_file():
    r=json.loads(RECEIPT.read_text(encoding='utf-8'))
    q=json.loads(QUALITY.read_text(encoding='utf-8'))
    existing_ok=(
        r.get('status')=='PASS' and
        q.get('status')=='PASS' and
        r.get('ticket_key_mismatches')==0 and
        q.get('ticket_key_mismatches')==0 and
        r.get('ticket_probability_catalog_sha256')==sha256(PROBS) and
        r.get('market_structure_sha256')==sha256(STRUCT) and
        r.get('quality_report_sha256')==sha256(QUALITY) and
        r.get('result_access') is False and
        r.get('settlement_access') is False and
        r.get('ev_roi_computed') is False and
        r.get('ECON_HOLDOUT1000')=='SEALED'
    )
    if existing_ok:
        print('✅ STAGE1 ALREADY PASS — 再計算不要')
        print('Probability SHA:',sha256(PROBS))
        print('Structure SHA:',sha256(STRUCT))
        print('Quality SHA:',sha256(QUALITY))
        print('Drive folder:',OUT)
        print('RESULT/PAYOUT/Settlement/EV/ROI access = none')
        print('ECON_HOLDOUT1000 = SEALED')
        STAGE1_ALREADY_COMPLETE=True
    else:
        STAGE1_ALREADY_COMPLETE=False
else:
    STAGE1_ALREADY_COMPLETE=False

if not STAGE1_ALREADY_COMPLETE:
    REPO=Path('/content/multiverse-research-stage1-pl-v2')
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])
    EXPECTED={
      'v3/historical_all_market/stage1_market_structure_only_v1.py':'23eba6b0423504e59bd7f74d6f22d88c387c8acf',
      'v3/historical_all_market/stage1_pl_ticket_probability_engine_v1.py':'029904b359e8151263c6a9bf8c4635f026941b0b',
    }
    for rel,exp in EXPECTED.items():
        obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
        if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
    print('✅ STAGE1 EXACT CODE BINDINGS PASS')
    PRICE=MY/'MULTIVERSE_ALL_MARKET_STAGE0_PRICE_RECOVERY_v2'/'PRICE_ONLY'/'DEV2000_ALL_MARKET_PRICE_CATALOGS_v2.jsonl'
    PRED=MY/'MULTIVERSE_DEV2000_PREDICTION_LOCK_v3_IPHONE_LITE'/'DEV2000_CANDIDATE_A_B1A_RECONSTITUTED_v1_PREDICTIONS.csv'
    OUT.mkdir(parents=True,exist_ok=True)
    for p in (STRUCT,PROBS,QUALITY,RECEIPT,LOG,ZIP):
        if p.exists(): p.unlink()
    cmd1=['python',str(REPO/'v3/historical_all_market/stage1_market_structure_only_v1.py'),str(PRICE),str(STRUCT)]
    cmd2=['python',str(REPO/'v3/historical_all_market/stage1_pl_ticket_probability_engine_v1.py'),str(PRED),str(STRUCT),str(PROBS),str(QUALITY)]
    logs=[]
    for label,cmd in [('STRUCTURE',cmd1),('PROBABILITY',cmd2)]:
        proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        logs.append(f'=== {label} ===\n{proc.stdout}\nRETURN={proc.returncode}\n')
        print(proc.stdout)
        if proc.returncode!=0:
            LOG.write_text('\n'.join(logs),encoding='utf-8')
            raise RuntimeError(f'FAIL-CLOSED Stage1 {label} return={proc.returncode}; see {LOG}')
    LOG.write_text('\n'.join(logs),encoding='utf-8')
    q=json.loads(QUALITY.read_text(encoding='utf-8'))
    if q.get('status')!='PASS' or q.get('ticket_key_mismatches')!=0:
        raise RuntimeError('FAIL-CLOSED Stage1 quality gate')
    if q.get('closing_odds_values_included') is not False or q.get('result_fields_included') is not False or q.get('settlement_fields_included') is not False or q.get('ev_roi_computed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage1 firewall quality flags')
    receipt={
      'record':'STAGE1_PL_RECEIPT_v1','status':'PASS',
      'market_structure_git_blob':'23eba6b0423504e59bd7f74d6f22d88c387c8acf',
      'pl_engine_git_blob':'029904b359e8151263c6a9bf8c4635f026941b0b',
      'prediction_sha256':sha256(PRED),'stage0_price_catalog_sha256':sha256(PRICE),
      'market_structure_sha256':sha256(STRUCT),'ticket_probability_catalog_sha256':sha256(PROBS),
      'quality_report_sha256':sha256(QUALITY),'races':q['races'],'output_rows':q['output_rows'],
      'ticket_key_mismatches':q['ticket_key_mismatches'],'scientific_trial_count':0,
      'result_access':False,'settlement_access':False,'ev_roi_computed':False,'ECON_HOLDOUT1000':'SEALED'
    }
    RECEIPT.write_text(json.dumps(receipt,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in (STRUCT,PROBS,QUALITY,RECEIPT,LOG): z.write(p,arcname=p.name)
    receipt['artifact_sha256']=sha256(ZIP)
    RECEIPT.write_text(json.dumps(receipt,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    print('✅ STAGE1 PASS')
    print(RECEIPT.read_text())
    print('成果物はDriveに保存済みです。自動ダウンロードは行いません。')
